# Modelo de Consumo Energético EV (Sensible a la Carga)

Este notebook valida la lógica de cálculo utilizada en el dashboard para la estimación del consumo eléctrico de la flota (kWh). 

## Fundamento Matemático

Utilizamos un modelo de **interpolación lineal** basado en la masa transportada (payload), siguiendo la filosofía del modelo GLEC v3.0 para emisiones de combustión, pero adaptado a ratios de eficiencia eléctrica (kWh/km).

In [1]:
import sys
import os
from pathlib import Path

# Configurar acceso al core del proyecto
project_root = Path(os.getcwd()).parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from logistic_core.config import EV_CONS_EMPTY, EV_CONS_FULL, VEHICLE_MAX_LOAD_KG

print(f"Coeficiente Vacío: {EV_CONS_EMPTY} kWh/km")
print(f"Coeficiente Plena Carga: {EV_CONS_FULL} kWh/km")
print(f"Capacidad Máxima: {VEHICLE_MAX_LOAD_KG:_} kg")

WARNING  GOOGLE_MAPS_API_KEY no está configurada. Se usará estimación Haversine.

Coeficiente Vacío: 1.05 kWh/km
Coeficiente Plena Carga: 1.7 kWh/km
Capacidad Máxima: 25_000 kg


### Función de Cálculo de Ratio

El ratio de consumo $C_{rate}$ se calcula como:

$$C_{rate} = C_{vacío} + (C_{lleno} - C_{vacío}) \cdot \frac{Carga}{Capacidad}$$

Donde:
- $C_{vacío}$: Consumo base del camión sin carga.
- $C_{lleno}$: Consumo a plena carga nominal (25t).

In [2]:
def get_ev_consumption_rate(load_kg):
    load_ratio = load_kg / VEHICLE_MAX_LOAD_KG
    return EV_CONS_EMPTY + (EV_CONS_FULL - EV_CONS_EMPTY) * load_ratio

# Ejemplo: Carga a la mitad (12,500 kg)
print(f"Consumo al 50% de carga: {get_ev_consumption_rate(12_500):.3f} kWh/km")

Consumo al 50% de carga: 1.375 kWh/km


### Simulación de una Ruta Estándar

Validamos el cálculo sobre una ruta típica de suministro y entrega:

In [3]:
tramos = [
    {"desc": "Depot -> Planta (Bobinas Papel)", "dist": 100, "carga": 25_000},
    {"desc": "Planta -> Cliente (Pallets Cartón)", "dist": 50, "carga": 15_000},
    {"desc": "Cliente -> Depot (Retorno Vacío)", "dist": 120, "carga": 0}
]

total_kwh = 0
total_dist = 0

print(f"{'-'*60}")
print(f"{'Tramo':<35} | {'Dist':<5} | {'Carga':<7} | {'kWh/km':<7} | {'Total'}")
print(f"{'-'*60}")

for t in tramos:
    rate = get_ev_consumption_rate(t['carga'])
    consumo = t['dist'] * rate
    total_kwh += consumo
    total_dist += t['dist']
    print(f"{t['desc']:<35} | {t['dist']:<5} | {t['carga']:<7} | {rate:<7.3f} | {consumo:.1f} kWh")

avg_efficiency = total_kwh / total_dist
print(f"{'-'*60}")
print(f"TOTAL RUTA: {total_kwh:.1f} kWh")
print(f"EFICIENCIA MEDIA (kWh/km): {avg_efficiency:.3f}")

------------------------------------------------------------
Tramo                               | Dist  | Carga   | kWh/km  | Total
------------------------------------------------------------
Depot -> Planta (Bobinas Papel)     | 100   | 25000   | 1.700   | 170.0 kWh
Planta -> Cliente (Pallets Cartón)  | 50    | 15000   | 1.440   | 72.0 kWh
Cliente -> Depot (Retorno Vacío)    | 120   | 0       | 1.050   | 126.0 kWh
------------------------------------------------------------
TOTAL RUTA: 368.0 kWh
EFICIENCIA MEDIA (kWh/km): 1.363


## Conclusiones para el TFM

1. **Precisión Física**: El modelo reconoce que el camión eléctrico de 44t requiere más energía en el tramo de suministro (bobinas de papel pesadas) que en el de entrega de cartón.
2. **Validación GLEC**: Aunque GLEC v3 no estandariza aún los kWh eléctricos por tramo, aplicamos el mismo rigor metodológico que para los combustibles fósiles, asegurando consistencia académica.